# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# jax.config.update('jax_disable_jit', True)

In [ ]:
import time
import functools
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.mp_mpl as mp_mpl

import lbfgs.lbfgs as lbfgs

In [ ]:
# general setup

T = 20.0  # s
num_steps = int(T / const.dt)
n = 200  # horizon

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.1])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

In [ ]:
# cost setup

weights = opt.ExpWeights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    omega=jnp.array([1e1, 1e1, 1e1]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
yaw_margins = [0.2 / 3.0, 0.1 / 3.0]
yaw_sizes = [2**0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    # margins=[0.2, 0.2, 0.2, 0.1],
    # sizes=[1.0, 2**3, 36.0, 2**7, 2**10],  # 36 == 2**5
    # margins=[0.4, 0.2, 0.1],
    # sizes=[1.0, 17.1, 33.5, 2**10],
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes = sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=yaw_margins,
    sizes=yaw_sizes,
    low=-const.max_rotary_yaw,
    high=const.max_rotary_yaw,
)
yaw_dot_cost = quartic_cost.QuarticCost.from_bounds(
    margins=yaw_margins,
    sizes=yaw_sizes,
    low=-const.max_rotary_vel,
    high=const.max_rotary_vel,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
    yaw_dot_cost=yaw_dot_cost,
)

In [ ]:
# train setup

def cost(
    args: tuple[
        opt.TrainState, opt.Weights, opt.CostTerms, jax.Array, jax.Array
    ],
    control_flat: jax.Array,
) -> jax.Array:
    train_state, weights, cost_terms, acc_ref, omega_ref = args
    return opt.cost_flat_jax(
        weights=weights,
        cost_terms=cost_terms,
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=train_state.rstate0,
        vstate0_irl=train_state.vstate0_irl,
        vstate0_sim=train_state.vstate0_sim,
        control0=train_state.control0,
        control_flat=control_flat,
        use_rotary=True,
    )

cost_and_grad = jax.jit(jax.value_and_grad(cost, argnums=1))
lbfgs_jit = jax.jit(lbfgs.lbfgs)

@functools.partial(jax.jit, static_argnames=["max_iter", "max_ls"])
def train_step_with_cost_jit(
    weights: opt.Weights,
    cost_terms: opt.CostTerms,
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    train_state: opt.TrainState,
    max_iter: int = 16,
    max_ls: int = 8,
) -> tuple[opt.TrainState, utils.TableSol, sci_opt.OptimizeResult]:
    ts = train_state
    opt_params = lbfgs.OptParamsLBFGS(
        fun=cost_and_grad,
        max_iter=max_iter,
        max_ls=max_ls,
        tol=1e-5,
        c1=1e-4,
        c2=0.9,
    )
    res = lbfgs_jit(
        opt_params=opt_params,
        x0=train_state.control_flat,
        fun_params=(ts, weights, cost_terms, acc_ref, omega_ref),
    )

    control = utils.Control.from_flat(res[0])
    rstate = utils.get_rstate(control, ts.rstate0)
    vstate_irl = utils.get_vstate_irl(
        rstate, control, ts.control0, ts.vstate0_irl
    )
    vstate_sim = utils.get_vstate(acc_ref, omega_ref, ts.vstate0_sim)
    table_sol = utils.TableSol(
        x=rstate,
        u=control,
        vstate_irl=vstate_irl,
        vstate_sim=vstate_sim,
        stats=utils.TableStats(
            time=jnp.squeeze(0.0),
            status=jnp.array(0),
            cost=jnp.array(res[1]),
        ),
    )

    next_state = opt.TrainState(
        rstate0=rstate.state[1],
        vstate0_irl=vstate_irl.x_state[0],  # NOT off-by-one
        vstate0_sim=vstate_sim.x_state[1],
        control0=control.control[0],
        control_flat=control.flatten(),
    )
    return next_state, table_sol, res

def train_step_with_cost(
    weights: opt.Weights,
    cost_terms: opt.CostTerms,
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    train_state: opt.TrainState,
    **kwargs,
) -> tuple[opt.TrainState, utils.TableSol, sci_opt.OptimizeResult, float]:
    t0 = time.time()
    opt_options = kwargs.get("opt_options", {"maxiter": 16, "maxls": 8})
    res = train_step_with_cost_jit(
        weights,
        cost_terms,
        acc_ref,
        omega_ref,
        train_state,
        opt_options["maxiter"],
        opt_options["maxls"],
    )
    res[0].control0.block_until_ready()
    t1 = time.time()
    return res[0], res[1], res[2], t1 - t0

train_step = functools.partial(train_step_with_cost, weights, cost_terms, acc_ref, omega_ref)

In [ ]:
# run setup
train_state = opt.TrainState.zero_init(n, vstate0_mode=("earth", "earth"))
tmp, _, _, _ = train_step(train_state)
print(tmp)
tmp.control_flat.block_until_ready()

# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     tmp, _, _, _ = train_step(train_state
#     tmp.control_flat.block_until_ready()

train_list = []
times = []
sol_list = []
res_list = []

In [ ]:
# run
train_step(train_state, opt_options={"maxiter": 4, "maxls": 1})  # pre-compile, for timing
for _ in tqdm.tqdm(range(num_steps)):
    train_state, sol, res, t_tot = train_step(train_state, opt_options={"maxiter": 4, "maxls": 1})
    train_list.append(train_state)
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
sol_list_end = []
extra_steps = 0

# sol_list_end = viz.split_tablesol(sol_list[-1])
# extra_steps = sol_list[-1].u.shape[0] - 1

trajectory = sol_list + sol_list_end
references= {
    "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(trajectory), 1)),
    "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(trajectory), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory)

In [ ]:
import pickle
rstate = {
    "trajectory": trajectory,
    "references": references,
    "weights": weights,
    "cost_terms": cost_terms,
}
with open("data/const_acc_400_state.pickle", "wb") as f:
    pickle.dump(rstate, f)

In [ ]:
# plot cost history
get_cost_ts = functools.partial(
    opt.get_scipy_cost_ts,
    weights,
    cost_terms,
    acc_ref,
    omega_ref,
    use_rotary=True,
)

costs = []
for state in train_list:
    cost_fun, _ = get_cost_ts(state)
    costs.append(cost_fun(np.array(state.control_flat)))

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.set_title("Cost history")
ax.set_xlabel("iter")
ax.set_ylabel("cost")
ax.plot(costs)
ax.grid()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
for i in range(6):
    ax.plot(sol_list[-1].u.control[:, i], label=f"{i}")
ax.grid()
ax.legend()
plt.show()

In [ ]:
# assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
trajectory = sol_list + sol_list_end
references={
    "xyz-acceleration": jnp.tile(
        A=acc_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
    ),
    "angular-velocity": jnp.tile(
        A=omega_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
    ),
}

In [ ]:
mp_mpl.call_mp_animate_trajectory(
    file_name="data/const_acc_3d.mp4",
    trajectory=trajectory,
)

In [ ]:
# mp_mpl.call_mp_animate_human_trajectory(
#     file_name="data/const_acc_human.mp4",
#     trajectory=trajectory,
#     references=references,
# )

In [ ]:
# reps = (len(trajectory), 1, 1)
# mp_mpl.call_mp_animate_cost_trajectory(
#     file_name="data/const_acc_cost.mp4",
#     acc_refs=jnp.tile(acc_ref, reps),
#     omega_refs=jnp.tile(omega_ref, reps),
#     weights=weights,
#     cost_terms=cost_terms,
#     trajectory=trajectory,
# )